# Baseline Nghiên cứu: Base 3B Zero-shot và PT 3B LoRA NTP trên TempUN Full Sạch (76 năm)

Notebook này triển khai và đánh giá mô hình **Qwen/Qwen2.5-3B-Instruct** trên toàn bộ 76 năm của tập dữ liệu **TempUN Full** (1947–2022):
- **Quy mô dữ liệu sạch**: 104,130 mẫu huấn luyện NWP và 5,292 mẫu đánh giá trắc nghiệm MCQA (6 bài toán: DB, MM, RB, TB, Rel, WB).
- **Đảm bảo tính trong sạch**: Tuyệt đối không chứa siêu dữ liệu FoNE hay chuỗi Can Chi được cấy sẵn vào dữ liệu thô.
- **Quy trình Khử thiên vị Chữ cái (Debiasing Protocol)**:
  * Academic Choice-Logit Scoring (Invalid Rate = 0.0%).
  * Hoán vị vòng tròn 4 phương án (ABCD, BCDA, CDAB, DABC) cho 4-choice MCQA và 2 chiều cho Yes/No.
  * Đo Debiased Mean Accuracy triệt tiêu letter bias.
  * Kiểm định thống kê ghép cặp McNemar đối chiếu.

In [ ]:
import os, sys, time, math, gzip, base64, json, random, gc, re
from typing import Optional, List, Dict, Any, Tuple, Union
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import stats

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f'GPU: {gpu_name} ({total_vram} GB VRAM)')
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    device = torch.device('cpu')
    print('WARNING: Running on CPU!')

SEEDS = [42]
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

# --- Cấu hình số bước huấn luyện ---
# MAX_TRAIN_STEPS = None: Chạy toàn bộ 1 epoch (104,130 mẫu ~ 6,508 steps hiệu dụng).
MAX_TRAIN_STEPS = None


In [ ]:
# --- Nạp và kiểm tra dữ liệu sạch TempUN Full ---
def load_dataset_records():
    train_path, eval_path = None, None
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidate_train = [p for p in kaggle_input.rglob('train_104k_clean*') if p.is_file()]
        candidate_eval = [p for p in kaggle_input.rglob('eval_5292_clean*') if p.is_file()]
        if candidate_train:
            train_path = sorted(candidate_train, key=lambda p: (not p.name.endswith('.gz'), len(str(p))))[0]
        if candidate_eval:
            eval_path = sorted(candidate_eval, key=lambda p: (not p.name.endswith('.gz'), len(str(p))))[0]
    
    if not train_path or not eval_path:
        local_roots = [Path('.'), Path('..'), Path('Data'), Path('../data'), Path('data')]
        for root in local_roots:
            if not root.exists():
                continue
            if not train_path:
                c_tr = [p for p in root.rglob('train_104k_clean*') if p.is_file()]
                if c_tr:
                    train_path = sorted(c_tr, key=lambda p: (not p.name.endswith('.gz'), len(str(p))))[0]
            if not eval_path:
                c_ev = [p for p in root.rglob('eval_5292_clean*') if p.is_file()]
                if c_ev:
                    eval_path = sorted(c_ev, key=lambda p: (not p.name.endswith('.gz'), len(str(p))))[0]
            if train_path and eval_path:
                break
    
    if not train_path or not eval_path:
        print('=== CHẨN ĐOÁN LỖI TÌM TỆP DỮ LIỆU ===')
        if kaggle_input.exists():
            print(f'Cấu trúc thư mục {kaggle_input}:')
            for root, dirs, files in os.walk(str(kaggle_input)):
                print(f'  Dir: {root} | Subdirs: {dirs} | Files: {files[:10]}')
        else:
            print(f'Không tồn tại thư mục {kaggle_input}. Thư mục hiện tại: {os.getcwd()}')
            print(f'Tệp tin trong CWD: {os.listdir(".")[:20]}')
        raise FileNotFoundError(
            f'Không tìm thấy tập dữ liệu (train_path={train_path}, eval_path={eval_path}). '
            'Vui lòng gắn dataset tnghuy11/tempun-full-clean vào kernel Kaggle.'
        )
    
    print(f'Đã định vị tập dữ liệu: train={train_path}, eval={eval_path}')
    
    def _read_records(p):
        if str(p).endswith('.gz'):
            with gzip.open(p, 'rt', encoding='utf-8') as f:
                return [json.loads(line) for line in f if line.strip()]
        else:
            with open(p, 'r', encoding='utf-8') as f:
                return [json.loads(line) for line in f if line.strip()]
                
    train_records = _read_records(train_path)
    eval_records = _read_records(eval_path)
    return train_records, eval_records

train_cases, eval_cases = load_dataset_records()
print(f'Đã nạp {len(train_cases)} mẫu huấn luyện NWP (Kỳ vọng: 104130).')
print(f'Đã nạp {len(eval_cases)} mẫu đánh giá MCQA (Kỳ vọng: 5292).')
assert len(train_cases) == 104130, f'Sai số lượng train: {len(train_cases)}'
assert len(eval_cases) == 5292, f'Sai số lượng eval: {len(eval_cases)}'
print('Kiểm tra dữ liệu: 100% khớp quy chuẩn sạch.')


In [ ]:
# --- Kiến trúc Toán học Ticktack (Han et al., 2025) & LoRA ---
def year_to_canchi_polar(years: torch.Tensor, alpha: float = 0.5, beta: float = 0.5) -> Tuple[torch.Tensor, torch.Tensor]:
    """Chuyển đổi năm sang toạ độ cực (theta, r) theo xoắn ốc Archimedes và chu kỳ Can Chi 60 năm (Eq. 2)."""
    cycle_indices = ((years.to(torch.int64) - 3) % 60).to(torch.float32)
    theta = (cycle_indices * 6.0) * (math.pi / 180.0)
    r = alpha + beta * theta
    return theta, r

def polar_to_cartesian(theta: torch.Tensor, r: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """Chiếu toạ độ cực sang mặt phẳng Descartes (x, y) (Eq. 3)."""
    x = (r * torch.cos(theta)).view(-1, 1)
    y = (r * torch.sin(theta)).view(-1, 1)
    return x, y

def compute_sinusoidal_te(coord: torch.Tensor, hidden_dim: int) -> torch.Tensor:
    """Tính mã hoá thời gian dạng Sinusoidal Eq. 4 trên toàn bộ hidden_dim."""
    device = coord.device
    dim_indices = torch.arange(0, hidden_dim, 2, device=device, dtype=torch.float32)
    inv_freq = 1.0 / (10000.0 ** (dim_indices / float(hidden_dim))).unsqueeze(0)
    coord_inp = coord * inv_freq
    return torch.stack([torch.sin(coord_inp), torch.cos(coord_inp)], dim=-1).flatten(start_dim=-2)

def compute_batch_te(
    years: Union[torch.Tensor, List[int], int],
    hidden_dim: int = 2048,
    scale_mode: str = 'raw',
    alpha: float = 0.5,
    beta: float = 0.5,
    device: Optional[Union[str, torch.device]] = None,
    dtype: torch.dtype = torch.float32,
    gating_factor: float = 0.01,
) -> torch.Tensor:
    """Tạo vector TE tổng hợp kích thước [B, 1, hidden_dim] chuẩn xác theo Han et al. (2025) và models.py."""
    if isinstance(years, (int, float)):
        years = torch.tensor([years], device=device or 'cpu')
    elif not isinstance(years, torch.Tensor):
        years = torch.tensor(years, device=device or 'cpu')

    if device is None:
        device = years.device
        
    years_dev = years.view(-1).to(device=device)
    theta, r = year_to_canchi_polar(years_dev, alpha=alpha, beta=beta)
    x, y = polar_to_cartesian(theta, r)
    x = x.to(device)
    y = y.to(device)
    pe_x = compute_sinusoidal_te(x, hidden_dim)
    pe_y = compute_sinusoidal_te(y, hidden_dim)
    te_raw = (pe_x + pe_y).unsqueeze(1).to(dtype=dtype)
    
    if scale_mode == 'raw':
        return te_raw
    elif scale_mode == 'scaled':
        scale = 1.0 / float(hidden_dim)
        return (te_raw * scale).to(dtype=dtype)
    elif scale_mode == 'gated':
        return (te_raw * gating_factor).to(dtype=dtype)
    else:
        raise ValueError(f'scale_mode không hợp lệ: {scale_mode}')

# --- Cấu trúc LoRALinear chuẩn hoá cho LoRA PEFT ---
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, alpha=16.0, base_layer=None):
        super().__init__()
        self.rank = rank
        self.scaling = alpha / rank if rank > 0 else 1.0
        self.base_layer = base_layer if base_layer is not None else nn.Linear(in_features, out_features, bias=False)
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False
            
        if rank > 0:
            self.lora_A = nn.Parameter(torch.empty(rank, in_features))
            self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            
    @classmethod
    def from_linear(cls, linear: nn.Linear, rank=8, alpha=16.0):
        mod = cls(linear.in_features, linear.out_features, rank=rank, alpha=alpha, base_layer=linear)
        mod.base_layer.weight.data.copy_(linear.weight.data)
        if linear.bias is not None:
            mod.base_layer.bias.data.copy_(linear.bias.data)
        return mod
        
    def forward(self, x):
        base_out = self.base_layer(x)
        if self.rank > 0:
            lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scaling
            return base_out + lora_out
        return base_out

def apply_lora_to_model(model, target_modules=['q_proj', 'v_proj'], rank=8, alpha=16.0):
    for p in model.parameters():
        p.requires_grad = False
    adapted = {}
    for name, module in model.named_modules():
        for target in target_modules:
            if name.endswith(target) and isinstance(module, nn.Linear):
                parent_name = name.rsplit('.', 1)[0] if '.' in name else ''
                child_name = name.rsplit('.', 1)[-1]
                parent = model.get_submodule(parent_name) if parent_name else model
                lora_layer = LoRALinear.from_linear(module, rank=rank, alpha=alpha)
                lora_layer.to(device=module.weight.device, dtype=module.weight.dtype)
                setattr(parent, child_name, lora_layer)
                adapted[name] = lora_layer
    return adapted

# --- Hàm mất mát tương phản Can Chi Intra-class ---
def compute_intra_loss(hidden_states: torch.Tensor, years: torch.Tensor) -> torch.Tensor:
    """Tính hàm mất mát tương phản chu kỳ Can Chi Eq. 8."""
    B = hidden_states.size(0)
    if B <= 1:
        return torch.tensor(0.0, device=hidden_states.device, dtype=hidden_states.dtype)
        
    h_pooled = hidden_states.mean(dim=1)
    h_norm = F.normalize(h_pooled, p=2, dim=-1)
    cos_sim = torch.mm(h_norm, h_norm.t())
    
    canchi_cycles = ((years.to(torch.int64) - 3) % 60).to(device=hidden_states.device)
    same_cycle = canchi_cycles.unsqueeze(1) == canchi_cycles.unsqueeze(0)
    eye_mask = torch.eye(B, dtype=torch.bool, device=hidden_states.device)
    pair_mask = same_cycle & (~eye_mask)
    
    if not pair_mask.any():
        return torch.tensor(0.0, device=hidden_states.device, dtype=hidden_states.dtype)
        
    intra_loss = (1.0 - cos_sim)[pair_mask].mean()
    return intra_loss


In [ ]:
# --- Khởi tạo mô hình Qwen2.5-3B-Instruct với tối ưu bộ nhớ --- 
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
HIDDEN_DIM = 2048

print(f'Đang tải tokenizer cho {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base_model():
    """Tải mô hình nền 3B ở độ chính xác nửa (fp16/bf16)."""
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    print(f'Nạp mô hình {MODEL_ID} (dtype={dtype})...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        if hasattr(model, 'enable_input_require_grads'):
            model.enable_input_require_grads()
        print('Đã kích hoạt Gradient Checkpointing (use_reentrant=False).')
    return model

def load_lora_model(rank=8, alpha=16.0):
    """Tải mô hình nền và áp dụng LoRA trên q_proj và v_proj với đóng băng 100% mô hình nền."""
    model = load_base_model()
    adapted = apply_lora_to_model(model, target_modules=['q_proj', 'v_proj'], rank=rank, alpha=alpha)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'LoRA đã thích ứng {len(adapted)} module. Số tham số huấn luyện: {trainable:,} / {total:,} ({trainable/total*100:.3f}%).')
    assert trainable < 5_000_000, f'Lỗi: Số tham số LoRA quá lớn ({trainable:,}), mô hình chưa được đóng băng đúng cách!'
    return model


In [ ]:
# --- Bộ dữ liệu Joint NWP Dataset & Tiêm mã hoá thời gian ---
class JointNWPDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=128):
        self.examples = []
        for s in samples:
            query = str(s.get('input', ''))
            ans = str(s.get('target', ''))
            year = int(s.get('year', 2000))
            full_text = f'{query.strip()} {ans.strip()}'
            enc = tokenizer(full_text, truncation=True, max_length=max_len, return_tensors='pt')
            input_ids = enc['input_ids'].squeeze(0)
            
            prefix_enc = tokenizer(query.strip(), truncation=True, max_length=max_len, return_tensors='pt')
            prefix_len = prefix_enc['input_ids'].size(1)
            
            labels = input_ids.clone()
            labels[:min(prefix_len, len(labels))] = -100
            
            self.examples.append({
                'input_ids': input_ids,
                'labels': labels,
                'year': year,
            })
            
    def __len__(self):
        return len(self.examples)
        
    def __getitem__(self, idx):
        return self.examples[idx]

def collate_joint_fn(batch, pad_token_id):
    max_len = max(len(x['input_ids']) for x in batch)
    b_in, b_lbl, b_yr = [], [], []
    for x in batch:
        pad_len = max_len - len(x['input_ids'])
        inp = torch.cat([x['input_ids'], torch.full((pad_len,), pad_token_id, dtype=torch.long)])
        lbl = torch.cat([x['labels'], torch.full((pad_len,), -100, dtype=torch.long)])
        b_in.append(inp)
        b_lbl.append(lbl)
        b_yr.append(x['year'])
    return {
        'input_ids': torch.stack(b_in),
        'labels': torch.stack(b_lbl),
        'years': torch.tensor(b_yr, dtype=torch.long),
    }

def find_date_token_mask(input_ids: torch.Tensor, years: torch.Tensor, tokenizer, device=None):
    """Xác định mặt nạ nhị phân [B, S, 1] cho đúng các token số năm trong câu hỏi."""
    if device is None:
        device = input_ids.device
    B, S = input_ids.shape
    mask = torch.zeros((B, S, 1), dtype=torch.bool, device=device)
    for b in range(B):
        yr_str = str(int(years[b].item()))
        yr_tokens = tokenizer.encode(yr_str, add_special_tokens=False)
        if not yr_tokens:
            continue
        k = len(yr_tokens)
        seq = input_ids[b].tolist()
        for i in range(S - k + 1):
            if seq[i:i + k] == yr_tokens:
                mask[b, i:i + k, 0] = True
    return mask

def inject_te_embeddings(model, tokenizer, input_ids: torch.Tensor, years: torch.Tensor, te_mode='broadcast', scale_mode='raw', device=None):
    """Tiêm vector thời gian TE vào biểu diễn token ban đầu."""
    if device is None:
        device = input_ids.device
    embed_tokens = model.get_input_embeddings()
    tok_emb = embed_tokens(input_ids)
    
    if te_mode == 'none':
        return tok_emb
        
    hidden_dim = tok_emb.size(-1)
    te = compute_batch_te(years, hidden_dim=hidden_dim, scale_mode=scale_mode, device=device, dtype=tok_emb.dtype)
    
    if te_mode == 'broadcast':
        return tok_emb + te
    elif te_mode == 'date_token':
        date_mask = find_date_token_mask(input_ids, years, tokenizer, device=device)
        return torch.where(date_mask, tok_emb + te, tok_emb)
    else:
        raise ValueError(f'te_mode không hợp lệ: {te_mode}')


In [ ]:
# --- Bộ sinh hoán vị phương án và Đánh giá Choice-Logit khử thiên vị ---
def generate_option_permutations(prompt_text: str, target_text: str):
    """Sinh hoán vị phương án cho cả trắc nghiệm 4 lựa chọn (4 vòng) và Yes/No (2 thứ tự)."""
    tgt = target_text.strip().lower()
    if tgt in ['yes', 'no']:
        p0 = prompt_text
        p1 = re.sub(r'Answer as Yes or No', 'Answer as No or Yes', prompt_text, flags=re.IGNORECASE)
        if p1 == p0:
            p1 = re.sub(r'Yes\s*(or|/)\s*No', 'No or Yes', prompt_text, flags=re.IGNORECASE)
        return [
            {'prompt': p0, 'target': tgt, 'rotation': 0, 'rot_name': 'Yes/No'},
            {'prompt': p1, 'target': tgt, 'rotation': 1, 'rot_name': 'No/Yes'}
        ]
        
    suffix = ''
    m_suffix = re.search(r'(\s*\n?\s*Answer:\s*)$', prompt_text, re.IGNORECASE)
    if m_suffix:
        suffix = m_suffix.group(1)
        base_prompt = prompt_text[:len(prompt_text) - len(suffix)]
    else:
        base_prompt = prompt_text
        
    pattern = re.compile(r'^(.*Options:\s*)\(a\)\s*(.*?)\s*\(b\)\s*(.*?)\s*\(c\)\s*(.*?)\s*\(d\)\s*(.*?)$', re.DOTALL)
    m = pattern.search(base_prompt)
    if not m:
        return [{'prompt': prompt_text, 'target': tgt, 'rotation': 0, 'rot_name': 'orig'}]
        
    prefix = m.group(1)
    opts = [m.group(2).strip(), m.group(3).strip(), m.group(4).strip(), m.group(5).strip()]
    letters = ['a', 'b', 'c', 'd']
    if tgt not in letters:
        return [{'prompt': prompt_text, 'target': tgt, 'rotation': 0, 'rot_name': 'orig'}]
        
    t_idx = letters.index(tgt)
    rot_names = ['ABCD', 'BCDA', 'CDAB', 'DABC']
    permutations = []
    for k in range(4):
        rot_opts = [opts[(j + k) % 4] for j in range(4)]
        rot_target = letters[(t_idx - k) % 4]
        rot_prompt = f'{prefix}(a) {rot_opts[0]} (b) {rot_opts[1]} (c) {rot_opts[2]} (d) {rot_opts[3]}{suffix}'
        permutations.append({
            'prompt': rot_prompt,
            'target': rot_target,
            'rotation': k,
            'rot_name': rot_names[k]
        })
    return permutations

def _extract_choice_token_ids(tokenizer, words):
    ids = []
    if tokenizer is not None and hasattr(tokenizer, 'encode'):
        for w in words:
            for prefix in ['', ' ']:
                try:
                    enc = tokenizer.encode(prefix + w, add_special_tokens=False)
                    if enc and len(enc) == 1:
                        ids.append(enc[0])
                except Exception:
                    pass
    return list(dict.fromkeys(ids))

def eval_choice_logit_single(model, tokenizer, prompt_text: str, target_text: str, year: int = None, te_mode: str = 'none', te_scale_mode: str = 'raw'):
    prompt = f'Question: {prompt_text.strip()}\nAnswer:' if not prompt_text.strip().endswith('Answer:') else prompt_text.strip()
    inputs = tokenizer(prompt, return_tensors='pt').to(device=model.device)
    
    with torch.no_grad():
        if te_mode == 'none':
            outputs = model(**inputs)
        else:
            yr_tensor = torch.tensor([year if year is not None else 2000], dtype=torch.long, device=model.device)
            inputs_embeds = inject_te_embeddings(model, tokenizer, inputs['input_ids'], yr_tensor, te_mode=te_mode, scale_mode=te_scale_mode, device=model.device)
            outputs = model(inputs_embeds=inputs_embeds, attention_mask=inputs.get('attention_mask'))
            
        logits = outputs.logits[0, -1, :]
        vocab_size = logits.size(-1)
        
    def _get_max_logit(token_candidates, fallback_ids):
        dynamic_ids = _extract_choice_token_ids(tokenizer, token_candidates)
        candidate_ids = dynamic_ids if dynamic_ids else fallback_ids
        valid_ids = [idx for idx in candidate_ids if 0 <= idx < vocab_size]
        if valid_ids:
            return max(logits[idx].item() for idx in valid_ids)
        return -1e9

    tgt = target_text.strip().lower()
    if tgt in ['yes', 'no']:
        scores = {
            'yes': _get_max_logit(['yes', 'Yes', 'YES'], [7414, 9454, 9834, 9693]),
            'no': _get_max_logit(['no', 'No', 'NO'], [2308, 2753, 902, 2152]),
        }
        pred = 'yes' if scores['yes'] >= scores['no'] else 'no'
    else:
        scores = {
            'a': _get_max_logit(['a', 'A'], [264, 362, 64, 32]),
            'b': _get_max_logit(['b', 'B'], [293, 425, 65, 33]),
            'c': _get_max_logit(['c', 'C'], [272, 356, 66, 34]),
            'd': _get_max_logit(['d', 'D'], [294, 422, 67, 35]),
        }
        pred = max(scores, key=scores.get)
        
    return {
        'pred': pred,
        'gold': tgt,
        'is_correct': (pred == tgt),
        'scores': {k: round(v, 4) for k, v in scores.items()},
    }

def eval_choice_logits_batch(model, tokenizer, prompt_texts, target_texts, year=None, te_mode='none', te_scale_mode='raw'):
    """Đánh giá theo lô các hoán vị phương án trong 1 forward pass duy nhất với left padding."""
    if not prompt_texts:
        return []
    if len(prompt_texts) != len(target_texts):
        raise ValueError(f'Mismatched batch lengths: {len(prompt_texts)} prompts vs {len(target_texts)} targets')
        
    dev = getattr(model, 'device', next(model.parameters()).device if hasattr(model, 'parameters') and list(model.parameters()) else torch.device('cpu'))
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    formatted_prompts = [
        f'Question: {p.strip()}\nAnswer:' if not p.strip().endswith('Answer:') else p.strip()
        for p in prompt_texts
    ]
    inputs = tokenizer(formatted_prompts, return_tensors='pt', padding=True).to(device=dev)
    
    with torch.no_grad():
        if te_mode == 'none':
            outputs = model(**inputs)
        else:
            B = len(prompt_texts)
            yr_val = year if year is not None else 2000
            yr_tensor = torch.full((B,), yr_val, dtype=torch.long, device=dev)
            inputs_embeds = inject_te_embeddings(
                model, tokenizer, inputs['input_ids'], yr_tensor,
                te_mode=te_mode, scale_mode=te_scale_mode, device=dev
            )
            outputs = model(inputs_embeds=inputs_embeds, attention_mask=inputs.get('attention_mask'))
            
        batch_logits = outputs.logits[:, -1, :]
        vocab_size = batch_logits.size(-1)
        
    choice_candidates = {
        'yes': (['yes', 'Yes', 'YES'], [7414, 9454, 9834, 9693]),
        'no': (['no', 'No', 'NO'], [2308, 2753, 902, 2152]),
        'a': (['a', 'A'], [264, 362, 64, 32]),
        'b': (['b', 'B'], [293, 425, 65, 33]),
        'c': (['c', 'C'], [272, 356, 66, 34]),
        'd': (['d', 'D'], [294, 422, 67, 35]),
    }
    resolved_ids = {}
    for opt, (words, fallback) in choice_candidates.items():
        dyn = _extract_choice_token_ids(tokenizer, words)
        cand = dyn if dyn else fallback
        resolved_ids[opt] = [idx for idx in cand if 0 <= idx < vocab_size]
        
    results = []
    for i, tgt_raw in enumerate(target_texts):
        log_i = batch_logits[i]
        tgt = tgt_raw.strip().lower()
        if tgt in ['yes', 'no']:
            scores = {
                'yes': max((log_i[idx].item() for idx in resolved_ids['yes']), default=-1e9),
                'no': max((log_i[idx].item() for idx in resolved_ids['no']), default=-1e9),
            }
            pred = 'yes' if scores['yes'] >= scores['no'] else 'no'
        else:
            scores = {
                'a': max((log_i[idx].item() for idx in resolved_ids['a']), default=-1e9),
                'b': max((log_i[idx].item() for idx in resolved_ids['b']), default=-1e9),
                'c': max((log_i[idx].item() for idx in resolved_ids['c']), default=-1e9),
                'd': max((log_i[idx].item() for idx in resolved_ids['d']), default=-1e9),
            }
            pred = max(scores, key=scores.get)
            
        results.append({
            'pred': pred,
            'gold': tgt,
            'is_correct': (pred == tgt),
            'scores': {k: round(v, 4) for k, v in scores.items()},
        })
    return results

def evaluate_dataset(model, tokenizer, eval_records, te_mode='none', te_scale_mode='raw', debias=True):
    model.eval()
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    case_results = {}
    canonical_correct = 0
    debiased_correct_sum = 0.0
    strict_correct = 0
    task_scores = defaultdict(lambda: {'canonical': 0, 'debiased_sum': 0.0, 'total': 0})
    
    t0 = time.time()
    total_cases = len(eval_records)
    for idx, case in enumerate(eval_records):
        cid = case.get('example_id', f'case_{idx}')
        task_id = case.get('task_id', 'Unknown')
        yr = case.get('year')
        
        if debias:
            perms = generate_option_permutations(case['input'], case['target'])
            p_prompts = [p['prompt'] for p in perms]
            p_targets = [p['target'] for p in perms]
            batch_res = eval_choice_logits_batch(
                model, tokenizer, p_prompts, p_targets,
                year=yr, te_mode=te_mode, te_scale_mode=te_scale_mode
            )
            perm_outcomes = [r['is_correct'] for r in batch_res]
            is_canonical = perm_outcomes[0]
            debiased_score = float(np.mean(perm_outcomes))
            is_strict = all(perm_outcomes)
        else:
            batch_res = eval_choice_logits_batch(
                model, tokenizer, [case['input']], [case['target']],
                year=yr, te_mode=te_mode, te_scale_mode=te_scale_mode
            )
            is_canonical = batch_res[0]['is_correct']
            debiased_score = 1.0 if is_canonical else 0.0
            is_strict = is_canonical
            
        if is_canonical:
            canonical_correct += 1
        debiased_correct_sum += debiased_score if debias else (1.0 if is_canonical else 0.0)
        if is_strict:
            strict_correct += 1
            
        task_scores[task_id]['canonical'] += int(is_canonical)
        task_scores[task_id]['debiased_sum'] += debiased_score
        task_scores[task_id]['total'] += 1
        
        case_key = f'{cid}#{idx}'
        case_results[case_key] = {
            'is_correct': is_canonical,
            'debiased_score': debiased_score,
            'is_strict': is_strict,
            'task_id': task_id,
        }
        
        if (idx + 1) % 500 == 0 or (idx + 1) == total_cases:
            elapsed = time.time() - t0
            rate = (idx + 1) / elapsed if elapsed > 0 else 0.0
            remaining = (total_cases - (idx + 1)) / rate if rate > 0 else 0.0
            print(f'Đã đánh giá: {idx + 1}/{total_cases} câu ({elapsed:.1f}s | {rate:.2f} câu/s | ETA: {remaining/60:.1f} phút)...')
            
    N = len(eval_records)
    task_summary = {}
    for t, v in sorted(task_scores.items()):
        cnt = v['total']
        task_summary[t] = {
            'canonical_acc': round(v['canonical'] / cnt * 100.0, 2) if cnt > 0 else 0.0,
            'debiased_mean_acc': round(v['debiased_sum'] / cnt * 100.0, 2) if cnt > 0 else 0.0,
            'count': cnt,
        }
        
    return {
        'canonical_acc': round(canonical_correct / N * 100.0, 2) if N > 0 else 0.0,
        'debiased_mean_acc': round(debiased_correct_sum / N * 100.0, 2) if N > 0 else 0.0,
        'strict_acc': round(strict_correct / N * 100.0, 2) if N > 0 else 0.0,
        'total': N,
        'task_summary': task_summary,
        'case_results': case_results,
    }


In [ ]:
# --- Vòng lặp huấn luyện LoRA tối ưu trên Tesla T4 (Micro-batch 8, Accumulation 2) ---
def train_baseline(
    cfg_name: str,
    train_data: list,
    te_mode: str = 'none',
    te_scale_mode: str = 'raw',
    sigma: float = 0.5,
    epochs: int = 1,
    lr: float = 2e-4,
    micro_batch_size: int = 8,
    gradient_accumulation_steps: int = 2,
    max_steps = None,
):
    print(f'\n========== BẮT ĐẦU HUẤN LUYỆN: {cfg_name} ==========')
    print(f'Tham số: te_mode={te_mode}, te_scale={te_scale_mode}, sigma={sigma}, epochs={epochs}')
    set_seed(42)
    
    model = load_lora_model(rank=8, alpha=16.0)
    model.train()
    
    train_dataset = JointNWPDataset(train_data, tokenizer, max_len=128)
    train_loader = DataLoader(
        train_dataset,
        batch_size=micro_batch_size,
        shuffle=True,
        collate_fn=lambda b: collate_joint_fn(b, pad_token_id=tokenizer.pad_token_id),
    )
    
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)
    
    step = 0
    total_loss_accum = 0.0
    t_start = time.time()
    
    for epoch in range(epochs):
        for b_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(model.device)
            labels = batch['labels'].to(model.device)
            years = batch['years'].to(model.device)
            
            if te_mode == 'none':
                outputs = model(input_ids=input_ids, labels=labels, output_hidden_states=(sigma > 0))
            else:
                inputs_embeds = inject_te_embeddings(
                    model, tokenizer, input_ids, years,
                    te_mode=te_mode, scale_mode=te_scale_mode, device=model.device
                )
                if inputs_embeds.is_leaf:
                    inputs_embeds.requires_grad_(True)
                outputs = model(inputs_embeds=inputs_embeds, labels=labels, output_hidden_states=(sigma > 0))
                
            loss_ntp = outputs.loss
            if sigma > 0 and hasattr(outputs, 'hidden_states') and outputs.hidden_states is not None:
                last_hidden = outputs.hidden_states[-1]
                loss_intra = compute_intra_loss(last_hidden, years)
                loss_total = loss_ntp + sigma * loss_intra
            else:
                loss_total = loss_ntp
                
            loss_total = loss_total / gradient_accumulation_steps
            loss_total.backward()
            total_loss_accum += loss_total.item() * gradient_accumulation_steps
            
            if (b_idx + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                optimizer.step()
                optimizer.zero_grad()
                step += 1
                
                if step % 250 == 0:
                    avg_l = total_loss_accum / 250.0
                    elapsed = time.time() - t_start
                    print(f'Epoch {epoch+1} | Step {step} | Loss trung bình: {avg_l:.4f} | Thời gian: {elapsed:.1f}s')
                    total_loss_accum = 0.0
                    
                if max_steps is not None and step >= max_steps:
                    print(f'Đã đạt ngưỡng max_steps={max_steps}, dừng huấn luyện sớm.')
                    break
                    
    print(f'Huấn luyện hoàn thành sau {time.time() - t_start:.1f} giây.')
    lora_state = {k: v.cpu() for k, v in model.state_dict().items() if 'lora_' in k}
    lora_filename = f'{cfg_name.lower()}_lora_weights.pt' if cfg_name else 'pt_3b_lora_weights.pt'
    torch.save(lora_state, lora_filename)
    if lora_filename != 'pt_3b_lora_weights.pt' and cfg_name and 'pt' in cfg_name.lower():
        torch.save(lora_state, 'pt_3b_lora_weights.pt')
    lora_size_mb = os.path.getsize(lora_filename) / (1024 * 1024)
    print(f'Đã lưu checkpoint LoRA ({len(lora_state)} tensors, {lora_size_mb:.2f} MB) vào {lora_filename}')
    return model


In [ ]:
# --- NẠP MỐC THAM CHIẾU BASE 3B ZERO-SHOT ĐÃ ĐƯỢC XÁC MINH VÀ HUẤN LUYỆN PT 3B ---
BENCHMARK_RESULTS = {}

BASE_3B_GZ_B64 = '''...'''

def load_verified_base_3b():
    if os.path.isfile('base_3b_benchmark_results.json'):
        with open('base_3b_benchmark_results.json', 'r', encoding='utf-8') as f:
            data = json.load(f)
            return data.get('Base_3B', data)
    raw = gzip.decompress(base64.b64decode(BASE_3B_GZ_B64))
    data = json.loads(raw.decode('utf-8'))
    base_res = data.get('Base_3B', data)
    with open('base_3b_benchmark_results.json', 'w', encoding='utf-8') as f:
        json.dump({'Base_3B': base_res}, f, indent=2)
    return base_res

# 1. Nạp mốc chuẩn Base 3B Zero-shot đã được đo đạc và xác minh an toàn
print('>>> [1/2] Nạp mốc chuẩn Base 3B Zero-shot (100% 5,292 mẫu)...')
res_base = load_verified_base_3b()
BENCHMARK_RESULTS['Base_3B'] = res_base
print(f'Base 3B (Verified) -> Canonical Acc: {res_base["canonical_acc"]}%, Debiased Mean: {res_base["debiased_mean_acc"]}%, Strict Consistency: {res_base["strict_acc"]}%')
print('Đã sẵn sàng kết quả Base 3B trong base_3b_benchmark_results.json')

# 2. PT 3B (LoRA NTP chuẩn)
print('\n>>> [2/2] Huấn luyện & Đánh giá PT 3B (LoRA NTP)...')
pt_model = train_baseline('PT_3B', train_cases, te_mode='none', sigma=0.0, epochs=1, max_steps=MAX_TRAIN_STEPS)
res_pt = evaluate_dataset(pt_model, tokenizer, eval_cases, te_mode='none', debias=True)
BENCHMARK_RESULTS['PT_3B'] = res_pt
print(f'PT 3B -> Canonical Acc: {res_pt["canonical_acc"]}%, Debiased Mean: {res_pt["debiased_mean_acc"]}%, Strict Consistency: {res_pt["strict_acc"]}%')
del pt_model
gc.collect()
torch.cuda.empty_cache()

# Lưu ngay checkpoint PT 3B ra đĩa
with open('pt_3b_benchmark_results.json', 'w', encoding='utf-8') as f:
    json.dump(BENCHMARK_RESULTS, f, indent=2)
print('Đã lưu an toàn checkpoint PT 3B vào pt_3b_benchmark_results.json')


In [ ]:
# --- TỔNG KẾT SO SÁNH VÀ KIỂM ĐỊNH MCNEMAR ---
print('\n' + '='*80)
print(f'{"Cấu hình":<15} | {"Canonical Acc":<15} | {"Debiased Mean Acc":<18} | {"Strict Consistency":<18}')
print('-'*80)
for name, res in BENCHMARK_RESULTS.items():
    print(f'{name:<15} | {res["canonical_acc"]:>13}% | {res["debiased_mean_acc"]:>16}% | {res["strict_acc"]:>16}%')
print('='*80)

def compute_mcnemar_table(cases_a, cases_b):
    b = sum(1 for cid in cases_a if cases_a[cid]['is_correct'] and not cases_b[cid]['is_correct'])
    c = sum(1 for cid in cases_a if not cases_a[cid]['is_correct'] and cases_b[cid]['is_correct'])
    a = sum(1 for cid in cases_a if cases_a[cid]['is_correct'] and cases_b[cid]['is_correct'])
    d = sum(1 for cid in cases_a if not cases_a[cid]['is_correct'] and not cases_b[cid]['is_correct'])
    stat = ((abs(b - c) - 1.0) ** 2) / (b + c) if (b + c) > 0 else 0.0
    pval = 2.0 * min(stats.binom.cdf(min(b, c), b + c, 0.5), 1.0 - stats.binom.cdf(min(b, c) - 1, b + c, 0.5)) if (b + c) > 0 else 1.0
    return {'b': b, 'c': c, 'a': a, 'd': d, 'stat': round(stat, 4), 'pval': round(pval, 6)}

if 'Base_3B' in BENCHMARK_RESULTS and 'PT_3B' in BENCHMARK_RESULTS:
    mcn_pt = compute_mcnemar_table(BENCHMARK_RESULTS['PT_3B']['case_results'], BENCHMARK_RESULTS['Base_3B']['case_results'])
    print(f'Kiểm định McNemar (PT_3B vs Base_3B): b={mcn_pt["b"]}, c={mcn_pt["c"]}, stat={mcn_pt["stat"]}, p={mcn_pt["pval"]}')

# Xuất kết quả tổng hợp ra file JSON
clean_export = {}
for k, v in BENCHMARK_RESULTS.items():
    clean_export[k] = {
        'canonical_acc': v['canonical_acc'],
        'debiased_mean_acc': v['debiased_mean_acc'],
        'strict_acc': v['strict_acc'],
        'task_summary': v['task_summary'],
    }
with open('pt_benchmark_summary.json', 'w', encoding='utf-8') as f:
    json.dump(clean_export, f, indent=2)
print('Đã lưu kết quả tổng hợp vào pt_benchmark_summary.json')
